# Phase 2(보강) — 참조 통계 적재 (reference_stats)

거시 통계(라오스 공식·전력)·경쟁사 단가/IR 등 **VOC가 아닌 시장 참조 지표**를 `reference_stats`에 단일 관리.
출처·신뢰도(official/igo/secondary/proposal) 표기 필수.

> 분석: `../outputs/laos_official_stats_analysis.md`, `../outputs/competitor_pricing_and_market_share.md`
> 총 38행 (laos_official 22 · laos_energy 7 · competitor_pricing 6 · competitor_ir 3)


In [ ]:
import sys; sys.path.append('../')
from src.db import get_connection, fetch_df
from psycopg2.extras import execute_values

## 1) 테이블 생성 (schema/create_tables.sql 와 동일)


In [ ]:
DDL = '''
CREATE TABLE IF NOT EXISTS reference_stats (
  id BIGSERIAL PRIMARY KEY,
  category VARCHAR(30) NOT NULL, entity VARCHAR(100) NOT NULL, metric VARCHAR(100) NOT NULL,
  value_num NUMERIC, value_text VARCHAR(200), unit VARCHAR(40), country CHAR(3),
  period VARCHAR(20) NOT NULL DEFAULT '-', source VARCHAR(200), reliability VARCHAR(10),
  note TEXT, created_at TIMESTAMPTZ DEFAULT NOW(),
  UNIQUE (category, entity, metric, period)
);
'''

## 2) 적재 데이터 (2026-06 수집)


In [ ]:
cols = ['category','entity','metric','value_num','value_text','unit','country','period','source','reliability','note']
rows = [
 ('laos_official', '등록차량_총', 'registered_vehicles', 3160000, None, '대', 'LAO', '2024', 'ASEANstats(LSB)', 'igo', '총 등록 도로차량'),
 ('laos_official', '등록차량_이륜차', 'registered_motorcycles', 2448254, None, '대', 'LAO', '2024', 'ASEANstats(LSB)', 'igo', '전체의 77.5%'),
 ('laos_official', 'EV', 'ev_penetration_rate', 0.14, None, '%', 'LAO', '2024', '계산(Customs/LSB)', 'secondary', '4,437 ÷ 316만'),
 ('laos_official', 'EV', 'ev_imports_cumulative', 4437, None, '대', 'LAO', '2024-10', 'Lao Customs(Laotian Times)', 'secondary', None),
 ('laos_official', 'EV', 'ev_sales_2023', 4631, None, '대', 'LAO', '2023', 'jclao/US-ASEAN', 'secondary', 'EV 판매=자동차2,592+이륜차2,039'),
 ('laos_official', 'EV', 'e_motorcycle_sales_2023', 2039, None, '대', 'LAO', '2023', 'jclao/US-ASEAN', 'secondary', '전기 이륜차 판매(2023)'),
 ('laos_official', 'EV', 'charging_stations', 41, None, '개소', 'LAO', '2023', '업계', 'secondary', '전국, 대부분 비엔티안'),
 ('laos_official', 'ICT', 'mobile_penetration', 85, None, '%', 'LAO', '2024', 'DataReportal(ITU/GSMA)', 'secondary', None),
 ('laos_official', 'ICT', 'internet_penetration', 66.2, None, '%', 'LAO', '2024', 'DataReportal(ITU)', 'secondary', None),
 ('laos_official', '인구', 'population_total', 7647000, None, '명', 'LAO', '2024', 'LSB LAOSIS(도별인구) 2024', 'official', 'World Bank 추계 7.95M'),
 ('laos_official', '인구', 'population_vientiane', 1029000, None, '명', 'LAO', '2024', 'LSB LAOSIS(도별인구)', 'official', '전국의 13.5%'),
 ('laos_official', '인구', 'households_total', 1372440, None, '가구', 'LAO', '2024', 'LSB LAOSIS', 'official', '전국 가구 수'),
 ('laos_official', 'Savannakhet', 'population_province', 1133000, None, '명', 'LAO', '2024', 'LSB LAOSIS', 'official', '최대 도 인구(SOM 2차 거점)'),
 ('laos_official', 'Champasack', 'population_province', 791000, None, '명', 'LAO', '2024', 'LSB LAOSIS', 'official', '남부 거점(팍세)'),
 ('laos_official', 'Luangprabang', 'population_province', 481000, None, '명', 'LAO', '2024', 'LSB LAOSIS', 'official', '관광 거점'),
 ('laos_official', '경제', 'gdp_per_capita', 2124, None, 'USD', 'LAO', '2024', 'World Bank', 'igo', None),
 ('laos_official', '경제', 'gdp_growth', 4, '~4', '%', 'LAO', '2024', 'World Bank', 'igo', None),
 ('laos_energy', '전기', 'electrification_rate', 96.5, None, '%', 'LAO', '2023', 'World Bank', 'igo', '가구 전기 보급률'),
 ('laos_energy', '전력', 'installed_capacity', 11600, None, 'MW', 'LAO', '2024', 'MEM(Lao)', 'secondary', '수력 81개 등 94개 발전소'),
 ('laos_energy', '전력', 'hydropower_share', 77, None, '%', 'LAO', '2023', 'IEA/저탄소', 'secondary', '발전 믹스 중 수력'),
 ('laos_energy', '전력', 'electricity_export', 40.8, None, 'TWh', 'LAO', '2025', '업계(power-tech 등)', 'secondary', '수입 0.6 TWh'),
 ('laos_energy', '전력', 'electricity_import', 0.6, None, 'TWh', 'LAO', '2025', '업계', 'secondary', None),
 ('laos_energy', '전력', 'export_revenue', 2.6, None, 'USD billion', 'LAO', '2024', '업계/정부', 'secondary', '전력 수출 수익'),
 ('laos_energy', '전기', 'residential_tariff', 647, None, 'KIP/kWh', 'LAO', '2024', 'EDL(Laotian Times)', 'secondary', '주택 평균 ~3US¢, 정부 보조'),
 ('competitor_pricing', 'LOCA EV', 'price_per_kwh', 4300, None, 'KIP/kWh', 'LAO', '2024', 'LOCA(정책안 비교)', 'secondary', '시간+kWh'),
 ('competitor_pricing', 'KOKKOK(제안)', 'price_per_kwh', 4000, None, 'KIP/kWh', 'LAO', '2026', '본 프로젝트 정책안', 'proposal', 'kWh 단일'),
 ('competitor_pricing', 'PTT blueplus+', 'price_per_kwh', None, '4.5~7.5', 'THB/kWh', 'THA', '2025', 'PTT 공식(TOU)', 'official', '피크/오프피크'),
 ('competitor_pricing', 'PEA VOLTA', 'price_per_kwh', None, '5.3~8.8', 'THB/kWh', 'THA', '2026', 'PEA 공식', 'official', '속도·시간 차등'),
 ('competitor_pricing', 'EleX by EGAT', 'price_per_kwh', 7.5, None, 'THB/kWh', 'THA', '2025', 'EGAT 공식', 'official', '정액'),
 ('competitor_pricing', 'V-Green', 'price_per_kwh', 3858, None, 'VND/kWh', 'VNM', '2024', 'VinFast 공식', 'official', '~2027 무료 프로모션'),
 ('competitor_ir', 'Grab', 'on_demand_gmv', 6.08, None, 'USD billion', 'MUL', 'Q4 2025', 'SEC 6-K', 'official', '+21% YoY'),
 ('competitor_ir', 'Grab', 'mtu', 50.5, None, 'million', 'MUL', 'Q4 2025', 'SEC 6-K', 'official', None),
 ('competitor_ir', 'GoTo', 'on_demand_gtv', 15.7, None, 'IDR trillion', 'MUL', 'Q1 2025', 'IDX IR', 'official', '모빌리티 5.9T'),
 ('laos_official','Vientiane Capital','households_province',187675,None,'가구','LAO','2024','LSB LAOSIS(도별가구)','official','전국 가구의 13.7%'),
 ('laos_official','Savannakhet','households_province',186447,None,'가구','LAO','2024','LSB LAOSIS(도별가구)','official','비엔티안과 거의 동률'),
 ('laos_official','Champasack','households_province',140231,None,'가구','LAO','2024','LSB LAOSIS(도별가구)','official','남부 거점'),
 ('laos_official','인구','age_15_39_share',43.2,None,'%','LAO','2024','LSB LAOSIS(성·연령)','official','EV 얼리어답터·드라이버·스마트폰 핵심층'),
 ('laos_official','인구','age_0_14_share',29.9,None,'%','LAO','2024','LSB LAOSIS(성·연령)','official','유소년 — 젊은 인구구조'),
]
print(len(rows), '행')

## 3) 업서트 (중복 시 갱신 — 재실행 안전)


In [ ]:
sql = f'''INSERT INTO reference_stats ({','.join(cols)}) VALUES %s
ON CONFLICT (category,entity,metric,period) DO UPDATE SET
 value_num=EXCLUDED.value_num, value_text=EXCLUDED.value_text, unit=EXCLUDED.unit,
 country=EXCLUDED.country, source=EXCLUDED.source, reliability=EXCLUDED.reliability, note=EXCLUDED.note'''
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(DDL)
        execute_values(cur, sql, rows)
    conn.commit()
print('✅ reference_stats upsert 완료')

## 4) 검증


In [ ]:
fetch_df('SELECT category, COUNT(*) n FROM reference_stats GROUP BY category ORDER BY category')